Import a diatomic xyz file and build parametrized Hamiltonian based on distance



In [22]:

from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.hamiltonians import ElectronicEnergy
from qiskit_nature.second_q.problems import ElectronicStructureProblem
from qiskit_nature.second_q.operators import FermionicOp
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD


In [23]:
symbols = ["H", "H"]
charge = 0
spin = 0  # 2S
basis = "sto-3g"

def H(x):
    """
    Function that builds parametrized qubit Hamiltonian based on distance x
    
    Uses qiskit nature
    """ 
    driver = PySCFDriver(
        atom=f"H 0 0 0; H 0 0 {x}",
        unit=DistanceUnit.ANGSTROM,
        charge=charge,
        spin=spin,
        basis=basis,
    )
    problem = driver.run()
    hamiltonian = problem.hamiltonian
    # get fermionic second-quantized operator
    second_q_op = hamiltonian.second_q_op()
    
    # Map to qubit operator
    mapper = JordanWignerMapper()
    qubit_p_op = mapper.map(second_q_op)
    return qubit_p_op


# For H2 with STO-3G: 2 electrons, 4 spin orbitals
num_particles = (1, 1)        # alpha, beta electrons
num_spin_orbitals = 4

mapper = JordanWignerMapper()

# 1) Hartree–Fock state |1100>
hf_init = HartreeFock(
    num_spatial_orbitals=num_spin_orbitals,
    num_particles=num_particles,
    qubit_mapper=mapper,
)

# 2) UCCSD ansatz with single + double excitations on top of HF
ansatz = UCCSD(
    num_spatial_orbitals=num_spin_orbitals,
    num_particles=num_particles,
    qubit_mapper=mapper,
    initial_state=hf_init,
)

print("Number of parameters:", ansatz.num_parameters)
print(ansatz)

Number of parameters: 15
     »
q_0: »
     »
q_1: »
     »
q_2: »
     »
q_3: »
     »
q_4: »
     »
q_5: »
     »
q_6: »
     »
q_7: »
     »
«     ┌──────────────────────────────────────────────────────────────────────────────────────────────┐
«q_0: ┤0                                                                                             ├
«     │                                                                                              │
«q_1: ┤1                                                                                             ├
«     │                                                                                              │
«q_2: ┤2                                                                                             ├
«     │                                                                                              │
«q_3: ┤3                                                                                             ├
«     │  EvolvedOps(t[0],t[1],t[

In [24]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit.library import UnitaryGate
from qiskit.primitives import Estimator

Ne = 2      # number of active electrons
Nq = 4     # number of active spin-orbitals / qubits

# --- Givens single-excitation gate (2 qubits) ---
def single_excitation_gate(theta):
    c, s = np.cos(theta), np.sin(theta)
    U = np.array(
        [
            [1, 0, 0, 0],
            [0, c, -s, 0],
            [0, s,  c, 0],
            [0, 0, 0, 1],
        ],
        dtype=complex,
    )
    return UnitaryGate(U, label="G")

# --- Givens double-excitation gate (4 qubits) ---
def double_excitation_gate(theta):
    """
    Acts as rotation between |1100> and |0011>
    in the computational basis (other states unchanged).
    """
    c, s = np.cos(theta), np.sin(theta)
    U = np.eye(16, dtype=complex)

    i = int("1100", 2)  # index of |1100>
    j = int("0011", 2)  # index of |0011>

    U[i, i] = c
    U[j, j] = c
    U[i, j] = -s
    U[j, i] = s

    return UnitaryGate(U, label="G2")

# --- Build variational circuit |Ψ(θ)⟩ ---
def build_ansatz(theta):
    """
    theta: array-like of length >= 2
    returns QuantumCircuit on Nq qubits
    """
    theta = np.array(theta, dtype=float)
    qc = QuantumCircuit(Nq)

    # 1) Hartree–Fock initialization: |1100...0>
    for q in range(Ne):
        qc.x(q)

    # 2) Example excitation pattern (you can adapt which qubits to couple)
    # single excitation between qubits 0 and 1
    qc.append(single_excitation_gate(theta[0]), [0, 1])

    # double excitation between qubits [0,1,2,3]
    qc.append(double_excitation_gate(theta[1]), [0, 1, 2, 3])

    return qc

# Draw example circuit
example_theta = np.array([0.1, 0.2])
example_circ = build_ansatz(example_theta)
print(example_circ.draw())


     ┌───┐┌────┐┌─────┐
q_0: ┤ X ├┤0   ├┤0    ├
     ├───┤│  G ││     │
q_1: ┤ X ├┤1   ├┤1    ├
     └───┘└────┘│  G2 │
q_2: ───────────┤2    ├
                │     │
q_3: ───────────┤3    ├
                └─────┘


In [25]:
import numpy as np
from qiskit.primitives import Estimator

def cost(theta, x):
    """g(theta, x) = <Psi(theta)| H(x) |Psi(theta)>"""
    # Build qubit Hamiltonian for geometry x
    op = H(x)
    # Build ansatz circuit for parameters theta (no Parameters → numeric gates)
    circ = build_ansatz(theta)

    # Use a fresh Estimator to avoid caching the previous result
    est = Estimator()
    job = est.run(circ, op)
    return float(job.result().values[0])

def grad_theta(theta, x, h=1e-2):
    """∂g/∂theta by central finite differences."""
    theta = np.array(theta, dtype=float)
    grad = np.zeros_like(theta)

    for i in range(len(theta)):
        t_plus = theta.copy()
        t_minus = theta.copy()
        t_plus[i] += h
        t_minus[i] -= h
        grad[i] = (cost(t_plus, x) - cost(t_minus, x)) / (2 * h)

    return grad

def grad_x(theta, x, h=1e-2):
    """∂g/∂x by central finite differences."""
    return (cost(theta, x + h) - cost(theta, x - h)) / (2 * h)

# Example: evaluate at initial guess
theta0 = np.array([20, 0.0])
x0 = 0.4  # Å

print("g(theta0, x0) =", cost(theta0, x0))
print("∂g/∂theta(theta0, x0) =", grad_theta(theta0, x0))
print("∂g/∂x(theta0, x0) =", grad_x(theta0, x0))


g(theta0, x0) = -1.0400089250289128


/tmp/ipykernel_1578015/3101139898.py:12: DeprecationWarning: The class ``qiskit.primitives.estimator.Estimator`` is deprecated as of qiskit 1.2. It will be removed no earlier than 3 months after the release date. All implementations of the `BaseEstimatorV1` interface have been deprecated in favor of their V2 counterparts. The V2 alternative for the `Estimator` class is `StatevectorEstimator`.
  est = Estimator()


∂g/∂theta(theta0, x0) = [0. 0.]
∂g/∂x(theta0, x0) = -1.0163659192695462


In [26]:
import numpy as np

def optimize_geometry(
    theta0,
    x0,
    lr_theta=0.02,
    lr_x=0.05,
    max_iters=50,
    tol_E=1e-6,
    tol_grad_theta=1e-3,
    tol_grad_x=1e-3,
    verbose=True,
):
    """
    Joint optimization of circuit parameters theta and geometry x.

    Simple gradient-descent loop using numerical gradients:
        theta_{k+1} = theta_k - lr_theta * ∂g/∂theta
        x_{k+1}     = x_k     - lr_x     * ∂g/∂x
    """
    theta = np.array(theta0, dtype=float)
    x = float(x0)
    history = []

    E_prev = None
    for it in range(max_iters):
        E = cost(theta, x)
        g_th = grad_theta(theta, x)
        g_x = grad_x(theta, x)

        norm_g_th = float(np.linalg.norm(g_th))
        history.append(
            {
                "iter": it,
                "E": E,
                "x": x,
                "grad_x": g_x,
                "norm_grad_theta": norm_g_th,
            }
        )

        if verbose:
            print(
                f"Iter {it:3d}: E = {E:.8f}, x = {x:.6f} Å, "
                f"|∂g/∂x| = {abs(g_x):.3e}, ||∂g/∂θ|| = {norm_g_th:.3e}"
            )

        # Convergence checks
        if E_prev is not None:
            if (
                abs(E - E_prev) < tol_E
                and abs(g_x) < tol_grad_x
                and norm_g_th < tol_grad_theta
            ):
                if verbose:
                    print("Converged.")
                break

        # Gradient-descent updates
        theta = theta - lr_theta * g_th
        x = x - lr_x * g_x

        E_prev = E

    return theta, x, history


# Example: run joint optimization from HF geometry and zero parameters
theta0 = np.zeros(2)
x0 = 0.6  # Å

theta_opt, x_opt, history = optimize_geometry(theta0, x0)
print("\nOptimized x* =", x_opt, "Å")
print("Optimized E* =", history[-1]["E"])

/tmp/ipykernel_1578015/3101139898.py:12: DeprecationWarning: The class ``qiskit.primitives.estimator.Estimator`` is deprecated as of qiskit 1.2. It will be removed no earlier than 3 months after the release date. All implementations of the `BaseEstimatorV1` interface have been deprecated in favor of their V2 counterparts. The V2 alternative for the `Estimator` class is `StatevectorEstimator`.
  est = Estimator()


Iter   0: E = -1.19292211, x = 0.600000 Å, |∂g/∂x| = 5.193e-01, ||∂g/∂θ|| = 0.000e+00
Iter   1: E = -1.20565963, x = 0.625966 Å, |∂g/∂x| = 4.625e-01, ||∂g/∂θ|| = 0.000e+00
Iter   2: E = -1.21579428, x = 0.649090 Å, |∂g/∂x| = 4.147e-01, ||∂g/∂θ|| = 0.000e+00
Iter   3: E = -1.22396760, x = 0.669825 Å, |∂g/∂x| = 3.742e-01, ||∂g/∂θ|| = 0.000e+00
Iter   4: E = -1.23064316, x = 0.688537 Å, |∂g/∂x| = 3.398e-01, ||∂g/∂θ|| = 0.000e+00
Iter   5: E = -1.23615940, x = 0.705525 Å, |∂g/∂x| = 3.101e-01, ||∂g/∂θ|| = 0.000e+00
Iter   6: E = -1.24076647, x = 0.721031 Å, |∂g/∂x| = 2.845e-01, ||∂g/∂θ|| = 0.000e+00
Iter   7: E = -1.24465159, x = 0.735256 Å, |∂g/∂x| = 2.621e-01, ||∂g/∂θ|| = 0.000e+00
Iter   8: E = -1.24795670, x = 0.748363 Å, |∂g/∂x| = 2.425e-01, ||∂g/∂θ|| = 0.000e+00
Iter   9: E = -1.25079078, x = 0.760489 Å, |∂g/∂x| = 2.252e-01, ||∂g/∂θ|| = 0.000e+00
Iter  10: E = -1.25323845, x = 0.771749 Å, |∂g/∂x| = 2.098e-01, ||∂g/∂θ|| = 0.000e+00
Iter  11: E = -1.25536620, x = 0.782240 Å, |∂g/∂x| = 1